In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
from SamplingMethods import Sampler_class

In [2]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelXG/ModelXG.json")

In [3]:
def SurrogateModelOfReality(n_ci, n_it):
    y_pred = loaded_model.predict(np.array([[n_ci],[n_it]]).T)[0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
class client(object):
    def __init__(self, df):
        self.df = df
    def summarize(self):
        return df

In [6]:
class RangeParameterConfig(object):
    def __init__(self, name, bounds):
        self.name = name
        self.bounds = bounds

In [7]:
class OptimisationSetup_class(object):
    def __init__(self):
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", bounds=(0, 1)),
            RangeParameterConfig(name="s2", bounds=(0, 1)),
            RangeParameterConfig(name="b1", bounds=(0, 1)),
        ]
OptimisationSetup_obj = OptimisationSetup_class()

In [8]:
y_max_lis = []

for i in range(100):
    sampler_obj = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler_obj.three.QuasirandomSampler3D_func(8,Parameters_lis).T
    y = []
    for row in X:
        s1 = row[0]
        s2 = row[1]
        b1 = row[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        y.append(SurrogateModelOfReality(n_ci,n_it))
    y = np.array(y)
    d = {"s1": X.T[0], "s2": X.T[1], "b1": X.T[2], "t1": y}
    df = pd.DataFrame(data=d)
    client_obj = client(df)
    # client_obj.summarize()
    sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client_obj)

    for _ in range(19):
        trial = sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client_obj)
        s1 = trial[0][0]
        s2 = trial[0][1]
        b1 = trial[0][2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        result = SurrogateModelOfReality(n_ci,n_it)
        nd = {"s1": [s1], "s2": [s2], "b1": [b1], "t1": [result]}
        df_new_rows = pd.DataFrame(data=nd)
        df = pd.concat([df,df_new_rows],ignore_index=True)
        client_obj = client(df)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client_obj.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

Trial 0 =========================================
14.017738342285156

Trial 1 =========================================
15.184355735778809

Trial 2 =========================================
14.174781799316406

Trial 3 =========================================
15.83945083618164

Trial 4 =========================================
14.346293449401855

Trial 5 =========================================
16.393735885620117

Trial 6 =========================================
15.220067024230957

Trial 7 =========================================
13.75991153717041

Trial 8 =========================================
14.219305992126465

Trial 9 =========================================
16.18293571472168

Trial 10 =========================================
16.6074275970459

Trial 11 =========================================
16.35771369934082

Trial 12 =========================================
14.219305992126465

Trial 13 =========================================
15.267313957214355

Trial 14 =============

In [9]:
y_max_arr = np.array(y_max_lis)
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.348304748535156
Avg = 15.091976194381713
Std = 0.8931341008999256


In [10]:
print(y_max_arr.tolist())

[14.017738342285156, 15.184355735778809, 14.174781799316406, 15.83945083618164, 14.346293449401855, 16.393735885620117, 15.220067024230957, 13.75991153717041, 14.219305992126465, 16.18293571472168, 16.6074275970459, 16.35771369934082, 14.219305992126465, 15.267313957214355, 14.312739372253418, 16.282541275024414, 14.834936141967773, 13.955521583557129, 13.893892288208008, 13.955521583557129, 16.273029327392578, 15.267313957214355, 17.348304748535156, 14.661030769348145, 16.6074275970459, 16.268138885498047, 13.955521583557129, 15.171805381774902, 14.181260108947754, 13.893892288208008, 13.955521583557129, 15.112234115600586, 14.187150001525879, 15.220067024230957, 15.079804420471191, 15.938945770263672, 14.249366760253906, 14.312739372253418, 15.171805381774902, 13.955521583557129, 15.171805381774902, 15.329530715942383, 16.35771369934082, 16.268138885498047, 16.393735885620117, 15.353317260742188, 15.337483406066895, 16.036523818969727, 15.83945083618164, 14.825654983520508, 14.266472

In [11]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [12]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    13.999640
1    14.187150
2    13.955522
3    15.768188
4    14.565559
..         ...
995  15.337483
996  15.184356
997  13.999640
998  13.955522
999  14.346293

[1000 rows x 1 columns]


In [13]:
# # Sanity check to make sure the MIPT is running correctly.
# df = client.summarize()
# types_lis = []
# for i in range(len(df)):
#     if i < 8:
#         types_lis.append("one-shot")
#     else:
#         types_lis.append("sequential")
# df["type"] = types_lis
# fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='type',width=1300, height=600)
# fig.show()